# Step 7: In-Sample vs Out-of-Sample Validation & Robustness Analysis

This notebook implements the quantitative validation, grid-search optimization, sensitivity sweep, and performance degradation analysis to evaluate strategy robustness and avoid overfitting.

In [ ]:
import os
import sys

# Insert project source root to system path for local imports
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.validation import split_dataset, optimize_parameters
from src.robustness import run_sensitivity_analysis, calculate_stability_score, calculate_generalization_score
from src.comparison import generate_comparison_table, plot_robustness_dashboard
from src.engine import backtest
from src.metrics import calculate_return_metrics, calculate_risk_adjusted_ratios
from src.risk import calculate_maximum_drawdown
from src.analytics import calculate_trade_statistics, calculate_benchmark_metrics

print("Modules imported successfully.")

## 1. Load Dataset & Split Chronologically

We partition our daily preprocessed Nifty 50 data:
* **In-Sample (IS)**: 2013-01-01 to 2019-12-31 (7 years, used for parameter optimization)
* **Out-of-Sample (OOS)**: 2020-01-01 to 2024-12-31 (5 years, locked for validation)

In [ ]:
prices = pd.read_parquet("../data/processed/nifty50_clean.parquet")

is_prices = split_dataset(prices, "2013-01-01", "2019-12-31")
oos_prices = split_dataset(prices, "2020-01-01", "2024-12-31")

print(f"In-Sample observations: {len(is_prices)}")
print(f"Out-of-Sample observations: {len(oos_prices)}")

## 2. Parameter Grid Search Optimization (In-Sample Only)

We define grid searches for both strategies and select the parameters that maximize the Sharpe Ratio.

In [ ]:
# Momentum Grid
mom_grid = []
for short_w in [10, 20, 30, 50]:
    for long_w in [50, 100, 150, 200]:
        if short_w < long_w:
            mom_grid.append({"short_window": short_w, "long_window": long_w})

# Mean Reversion Grid
mr_grid = []
for w in [10, 20, 30, 50]:
    for entry_t in [-1.5, -2.0, -2.5]:
        for exit_t in [-0.25, -0.5, -1.0]:
            mr_grid.append({"window": w, "entry_threshold": entry_t, "exit_threshold": exit_t})

print(f"Momentum parameter sets: {len(mom_grid)}")
print(f"Mean Reversion parameter sets: {len(mr_grid)}")

In [ ]:
print("Running Momentum optimization sweep...")
mom_opt_results = optimize_parameters(is_prices, "momentum", mom_grid, objective="max_sharpe")
display(mom_opt_results.head(5))

print("\nRunning Mean Reversion optimization sweep...")
mr_opt_results = optimize_parameters(is_prices, "mean_reversion", mr_grid, objective="max_sharpe")
display(mr_opt_results.head(5))

## 3. Freeze Best Parameters & Backtest on Unseen OOS Period

In [ ]:
# Extract frozen parameters
best_mom_params = mom_opt_results.iloc[0].to_dict()
best_mr_params = mr_opt_results.iloc[0].to_dict()

print("Frozen Momentum Parameters:", {k: best_mom_params[k] for k in ["short_window", "long_window"]})
print("Frozen Mean Reversion Parameters:", {k: best_mr_params[k] for k in ["window", "entry_threshold", "exit_threshold"]})

## 4. Run Backtests & Compile Performance Comparison Tables

In [ ]:
# Run Momentum Backtests
from src.momentum import MomentumSignalGenerator
mom_gen = MomentumSignalGenerator(short_window=int(best_mom_params["short_window"]), long_window=int(best_mom_params["long_window"]))
mom_is_sig = mom_gen.generate_signals(is_prices)
mom_oos_sig = mom_gen.generate_signals(oos_prices)

mom_is_backtest = backtest(is_prices, mom_is_sig["Raw_Signal"])
mom_oos_backtest = backtest(oos_prices, mom_oos_sig["Raw_Signal"])

# Run Mean Reversion Backtests
from src.mean_reversion import MeanReversionSignalGenerator
mr_gen = MeanReversionSignalGenerator(window=int(best_mr_params["window"]), entry_threshold=float(best_mr_params["entry_threshold"]), exit_threshold=float(best_mr_params["exit_threshold"]))
mr_is_sig = mr_gen.generate_signals(is_prices)
mr_oos_sig = mr_gen.generate_signals(oos_prices)

mr_is_backtest = backtest(is_prices, mr_is_sig["Raw_Signal"])
mr_oos_backtest = backtest(oos_prices, mr_oos_sig["Raw_Signal"])

In [ ]:
# Helper function to gather metrics dict
def gather_metrics(backtest_res, benchmark_returns):
    ret_m = calculate_return_metrics(backtest_res.portfolio["Portfolio_Value"], backtest_res.portfolio["Daily_Return"])
    bench_m = calculate_benchmark_metrics(backtest_res.portfolio["Daily_Return"], benchmark_returns)
    risk_m = calculate_risk_adjusted_ratios(backtest_res.portfolio["Daily_Return"], backtest_res.portfolio["Portfolio_Value"], benchmark_returns, bench_m["Beta"])
    trade_m = calculate_trade_statistics(backtest_res.trade_book)
    max_dd, _, _, _ = calculate_maximum_drawdown(backtest_res.portfolio["Portfolio_Value"])
    return {**ret_m, **bench_m, **risk_m, **trade_m, "Max_Drawdown": max_dd * 100.0, "Volatility": backtest_res.portfolio["Daily_Return"].std() * np.sqrt(252)}

bench_returns_is = is_prices["Close"].pct_change().fillna(0.0)
bench_returns_oos = oos_prices["Close"].pct_change().fillna(0.0)

mom_is_m = gather_metrics(mom_is_backtest, bench_returns_is)
mom_oos_m = gather_metrics(mom_oos_backtest, bench_returns_oos)
mr_is_m = gather_metrics(mr_is_backtest, bench_returns_is)
mr_oos_m = gather_metrics(mr_oos_backtest, bench_returns_oos)

print("=== Momentum Side-by-Side Comparison ===")
mom_comp = generate_comparison_table(mom_is_m, mom_oos_m)
display(mom_comp)

print("\n=== Mean Reversion Side-by-Side Comparison ===")
mr_comp = generate_comparison_table(mr_is_m, mr_oos_m)
display(mr_comp)

## 5. Robustness & Sensitivity Analysis

We evaluate neighborhood parameters to check for overfitting and compute stability scores.

In [ ]:
print("Running Momentum parameter sensitivity analysis...")
mom_sens = run_sensitivity_analysis(is_prices, "momentum", {"short_window": int(best_mom_params["short_window"]), "long_window": int(best_mom_params["long_window"])})
mom_stability = calculate_stability_score(mom_sens)
print(f"Momentum Stability Score: {mom_stability:.2f}%")

print("\nRunning Mean Reversion parameter sensitivity analysis...")
mr_sens = run_sensitivity_analysis(is_prices, "mean_reversion", {"window": int(best_mr_params["window"]), "entry_threshold": float(best_mr_params["entry_threshold"]), "exit_threshold": float(best_mr_params["exit_threshold"])})
mr_stability = calculate_stability_score(mr_sens)
print(f"Mean Reversion Stability Score: {mr_stability:.2f}%")

## 6. Plot Robustness Dashboards (20 charts per strategy)

In [ ]:
output_figures_dir = "../reports/figures/"

plot_robustness_dashboard(
    "momentum",
    mom_is_backtest,
    mom_oos_backtest,
    mom_opt_results,
    mom_sens,
    prices,
    output_dir=output_figures_dir
)

plot_robustness_dashboard(
    "mean_reversion",
    mr_is_backtest,
    mr_oos_backtest,
    mr_opt_results,
    mr_sens,
    prices,
    output_dir=output_figures_dir
)

print("All 40 robustness and validation charts successfully plotted and saved to reports/figures!")